# 02 - Dataset 1 (EMSCAD): modelling setup

This notebook sets up everything the supervised models share: the data, the feature code, the cross-validation folds and the scoring functions. No model comparison is run here. Each model notebook starts from this setup so that all models are scored on the same folds with the same metrics.

**Sections**
1. Feature engineering in brief
2. Imports and data
3. Cross-validation setup
4. Scoring and evaluation functions
5. Check the setup with the logistic-regression baseline
6. Test-set switch

## 1. Feature engineering in brief

Notebook 01 produced the feature code now in `src/features.py`. In short:

- **Split.** Postings are grouped by near-duplicate text and company profile. Whole groups stay on one side of every split and every cross-validation fold.
- **Row-wise features** (39) need no fitting: missing-field flags and count, masked contact flags, salary parsing, country, and text-style statistics. They are saved in `d1_train_feature_engineered.csv` and `d1_test_feature_engineered.csv`.
- **Learned features** are rebuilt inside each fold from the training part only: one-hot categories, TF-IDF (1 to 2-grams, up to 30,000 terms) and the fraud-phrase score.
- **Feature sets.** A0 = raw fields, A1 = + engineered, A2 = + text style + TF-IDF, A3 = + fraud-phrase score. **All models use A2** (`PRIMARY_SET`): TF-IDF carries the signal, the structured features are cheap and interpretable, and the phrase score added nothing measurable (+0.016 PR-AUC, within noise) while being the slowest part to compute. The A0–A3 ladder stays in the report as the feature ablation, run once with logistic regression.
- **Tree and boosting models** can pass `tfidf_max_features=2000` to `make_features` to keep the sparse text block small.
- **Findings carried forward.** TF-IDF carries nearly all the detectable signal, and TF-IDF-only logistic regression (PR-AUC about 0.58 in grouped CV) is the baseline to beat. No block was shown to beat TF-IDF alone, and differences below about 0.1 PR-AUC cannot be resolved with roughly 331 independent fraud groups.
- **Missing values** are kept informative (flags and a count), not imputed away. Dense columns are standardised; tree models may use the same inputs unscaled.

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.metrics import (average_precision_score, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedGroupKFold

try:
    from IPython.display import display
except ImportError:
    display = print

# Find the project folder (the one that contains data/processed/d1_train.csv) and make src/ importable.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data" / "processed" / "d1_train.csv").exists()), Path.cwd())
sys.path.insert(0, str(ROOT))

from src.features import (FEATURE_SETS, GROUP, PRIMARY_SET, SEED, TARGET, build_inputs, load_d1,
                          logistic_pipeline, make_features)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)

PROCESSED_DIR = ROOT / "data" / "processed"
FIG_DIR = ROOT / "reports" / "figures"
TAB_DIR = ROOT / "reports" / "tables"
for folder in (FIG_DIR, TAB_DIR):
    folder.mkdir(parents=True, exist_ok=True)

train, test = load_d1(PROCESSED_DIR)
X_train, X_test = build_inputs(train), build_inputs(test)     # tables the feature pipelines read
y_train, y_test = train[TARGET].to_numpy(), test[TARGET].to_numpy()
groups_train = train[GROUP].to_numpy()

print("Project folder:", ROOT)
print(f"train: {X_train.shape} | fraud {y_train.mean():.2%} ({y_train.sum()}) | groups {train[GROUP].nunique():,}")
print(f"test : {X_test.shape} | fraud {y_test.mean():.2%} ({y_test.sum()}) | groups {test[GROUP].nunique():,}")

## 3. Cross-validation setup

**Design.** Repeated 5-fold `StratifiedGroupKFold` on the training set only. Groups are kept whole, and the fraud rate is kept about equal across folds. Repeating with different seeds gives several estimates of each model, which helps because a single 5-fold run is noisy with so few fraud groups.

**Same folds for every model.** The folds are built once, here, and stored in `FOLDS`. Models must use `FOLDS`, never create their own, so that scores are paired fold by fold and can be compared with the corrected paired interval in section 4.

**Where tuning goes.** Choosing hyperparameters or a threshold on the same folds that report the score is optimistic. When a model has hyperparameters to tune, use an inner grouped split inside each training fold (nested CV), and report only the outer scores.

In [ ]:
N_SPLITS = 5
N_REPEATS = 5            # use 2 for quick experiments, 5 for reported results


def make_folds(groups, y, n_splits=N_SPLITS, n_repeats=N_REPEATS, seed=SEED):
    """List of (repeat, fold, fit_idx, val_idx). Whole groups stay together in every fold."""
    folds = []
    for r in range(n_repeats):
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed + r)
        for k, (fit_idx, val_idx) in enumerate(splitter.split(np.zeros(len(y)), y, groups)):
            folds.append((r, k, fit_idx, val_idx))
    return folds


FOLDS = make_folds(groups_train, y_train)

# Safety checks: no group in both sides, every row validated exactly once per repeat
for r, k, fit_idx, val_idx in FOLDS:
    assert set(groups_train[fit_idx]).isdisjoint(groups_train[val_idx])
for r in range(N_REPEATS):
    seen = np.concatenate([v for rr, _, _, v in FOLDS if rr == r])
    assert len(seen) == len(y_train) and len(np.unique(seen)) == len(y_train)

fold_table = pd.DataFrame(
    [{"repeat": r, "fold": k, "train_rows": len(f), "val_rows": len(v),
      "val_fraud": int(y_train[v].sum()), "val_fraud_rate": y_train[v].mean(),
      "val_groups": len(np.unique(groups_train[v]))} for r, k, f, v in FOLDS])
print(f"{len(FOLDS)} folds ({N_REPEATS} repeats x {N_SPLITS})")
display(fold_table.groupby("repeat")[["val_rows", "val_fraud", "val_fraud_rate"]].agg(["min", "max"]).round(3))

## 4. Scoring and evaluation functions

**Metrics.** The fraud class is rare (about 5%), so accuracy is not reported. Each model is scored with:
- **PR-AUC** (average precision) for the fraud class, the main metric. A model with no skill scores about the fraud rate.
- **ROC AUC**, for reference.
- **Precision, recall and F1** at a chosen threshold.
- **Review budget**: precision and recall if only the top 5% of postings by score are reviewed.

**Thresholds.** `score_summary` can pick the F1-maximising threshold on the predictions it is given. That is optimistic when the same predictions are used for scoring. For honest results, fix the threshold on the training part of each fold and apply it to the validation part, which `cross_validate` does when `threshold="inner"`.

**Fold-level and pooled scores.** Fold-level PR-AUC feeds the paired comparison. Pooled out-of-fold predictions (one per repeat) give the precision-recall curves and the pooled metrics.

In [ ]:
def best_f1_threshold(y, p):
    precision, recall, thresholds = precision_recall_curve(y, p)
    f1 = 2 * precision[:-1] * recall[:-1] / np.clip(precision[:-1] + recall[:-1], 1e-12, None)
    return thresholds[np.argmax(f1)]


def score_summary(y, p, threshold=None, review_share=0.05):
    """Fraud-class metrics for predicted probabilities p. If no threshold is given, the F1-maximising
    threshold on these same predictions is used (optimistic)."""
    if threshold is None:
        threshold = best_f1_threshold(y, p)
    flagged = p >= threshold
    top = np.argsort(-p)[:max(int(review_share * len(p)), 1)]
    return {"pr_auc": average_precision_score(y, p), "roc_auc": roc_auc_score(y, p), "threshold": threshold,
            "precision": precision_score(y, flagged, zero_division=0), "recall": recall_score(y, flagged),
            "f1": f1_score(y, flagged, zero_division=0), "share_flagged": flagged.mean(),
            "precision_top5pct": y[top].mean(), "recall_top5pct": y[top].sum() / max(y.sum(), 1)}


def cross_validate(make_model, X=None, y=None, folds=None, threshold="inner", verbose=True):
    """Fit a fresh model on every fold and score the held-out part.

    make_model : function returning an unfitted model or pipeline (it is called once per fold, so
                 learned features are refitted on each training part). It must accept the table X.
    threshold  : "inner" fixes the F1-maximising threshold on the training part of the fold (an
                 in-sample choice, still slightly optimistic for flexible models) and applies it to the
                 validation part; None skips threshold metrics.

    Returns (fold_scores, oof), where fold_scores is a DataFrame with one row per fold and oof is an
    array (n_repeats, n_rows) of out-of-fold probabilities.
    """
    X = X_train if X is None else X
    y = y_train if y is None else y
    folds = FOLDS if folds is None else folds
    n_repeats = max(r for r, *_ in folds) + 1
    oof = np.zeros((n_repeats, len(y)))
    rows, t0 = [], time.time()
    for r, k, fit_idx, val_idx in folds:
        model = make_model()
        model.fit(X.iloc[fit_idx], y[fit_idx])
        p_val = model.predict_proba(X.iloc[val_idx])[:, 1]
        oof[r, val_idx] = p_val
        row = {"repeat": r, "fold": k, "pr_auc": average_precision_score(y[val_idx], p_val),
               "roc_auc": roc_auc_score(y[val_idx], p_val)}
        if threshold == "inner":
            thr = best_f1_threshold(y[fit_idx], model.predict_proba(X.iloc[fit_idx])[:, 1])
            s = score_summary(y[val_idx], p_val, threshold=thr)
            row.update({m: s[m] for m in ("precision", "recall", "f1", "precision_top5pct", "recall_top5pct")})
        rows.append(row)
    if verbose:
        print(f"{len(folds)} folds in {(time.time() - t0) / 60:.1f} min")
    return pd.DataFrame(rows), oof


def summarise(fold_scores, name):
    """One row of mean and spread across folds."""
    cols = [c for c in fold_scores.columns if c not in ("repeat", "fold")]
    out = {}
    for c in cols:
        out[c] = fold_scores[c].mean()
    out["pr_auc_sd"] = fold_scores["pr_auc"].std(ddof=1)
    return pd.Series(out, name=name)


def pooled_metrics(oof, y=None):
    """Pooled out-of-fold metrics, averaged over repeats (threshold chosen on the pooled predictions,
    so optimistic; use for curves and rough comparison, not for the reported precision/recall)."""
    y = y_train if y is None else y
    return pd.DataFrame([score_summary(y, p) for p in oof]).mean()


def paired_comparison(scores_a, scores_b, n_splits=N_SPLITS):
    """Mean fold-by-fold PR-AUC difference (a minus b) with an approximate 95% interval using the
    Nadeau-Bengio correction for overlapping training sets. Both inputs must come from the same FOLDS."""
    d = np.asarray(scores_a["pr_auc"]) - np.asarray(scores_b["pr_auc"])
    se = np.sqrt((1 / len(d) + 1 / (n_splits - 1)) * d.var(ddof=1))
    return {"mean_diff": d.mean(), "se": se, "low": d.mean() - 2 * se, "high": d.mean() + 2 * se,
            "share_folds_a_better": (d > 0).mean()}


def plot_pr_curves(oofs, path=None):
    """Precision-recall curves from the first repeat's out-of-fold predictions. oofs: {name: oof array}."""
    fig, ax = plt.subplots(figsize=(5.4, 4.2))
    for name, oof in oofs.items():
        precision, recall, _ = precision_recall_curve(y_train, oof[0])
        ax.plot(recall, precision, label=f"{name} (PR-AUC {average_precision_score(y_train, oof[0]):.2f})")
    ax.axhline(y_train.mean(), color="#c0504d", linestyle="--", linewidth=1, label="no skill")
    ax.set_xlabel("Recall (fraud class)")
    ax.set_ylabel("Precision (fraud class)")
    ax.set_title("Out-of-fold precision-recall curves")
    ax.legend(frameon=False)
    plt.tight_layout()
    if path:
        plt.savefig(path, dpi=200)
    plt.show()

## 5. Check the setup with the logistic-regression baseline

A single run of the TF-IDF-only baseline and of logistic regression on the primary set (A2) confirms that the folds, features and scoring work together, and stores the two reference results that every later model is compared with.

In [ ]:
RESULTS = {}


def run_model(name, make_model, **kwargs):
    """Cross-validate a model on the shared folds and store its results."""
    fold_scores, oof = cross_validate(make_model, **kwargs)
    RESULTS[name] = {"folds": fold_scores, "oof": oof}
    return summarise(fold_scores, name)


baseline = run_model("LogReg, TF-IDF only", lambda: logistic_pipeline(["tfidf"]))
primary = run_model(f"LogReg, {PRIMARY_SET}", lambda: logistic_pipeline(FEATURE_SETS[PRIMARY_SET]))
display(pd.concat([baseline, primary], axis=1).T.round(3))
print(f"No-skill PR-AUC is about the fraud rate: {y_train.mean():.3f}")

In [ ]:
def results_table():
    """Summary of every model stored in RESULTS, best PR-AUC first."""
    return pd.DataFrame([summarise(v["folds"], k) for k, v in RESULTS.items()]).sort_values("pr_auc", ascending=False)


def compare_to(reference, names=None):
    """Paired comparison of each stored model against a reference model (same folds)."""
    names = [n for n in (names or RESULTS) if n != reference]
    return pd.DataFrame({n: paired_comparison(RESULTS[n]["folds"], RESULTS[reference]["folds"]) for n in names}).T


display(results_table().round(3))
plot_pr_curves({k: v["oof"] for k, v in RESULTS.items()}, FIG_DIR / "model_pr_curves.png")

## 6. Test-set switch

The test file is for one final evaluation, after the model, features and threshold are all fixed. The switch stays `False` while models are being developed. When a final model is chosen, fit it on all training rows, fix the threshold from the training out-of-fold predictions, score the test file once, and change nothing afterwards.

In [ ]:
EVALUATE_ON_TEST = False      # turn on once, at the very end

def evaluate_on_test(make_model, threshold):
    """Fit on all training rows and score the test file once. The threshold must be fixed beforehand."""
    model = make_model().fit(X_train, y_train)
    p = model.predict_proba(X_test)[:, 1]
    return pd.Series(score_summary(y_test, p, threshold=threshold))

if EVALUATE_ON_TEST:
    thr = best_f1_threshold(y_train, RESULTS["LogReg, TF-IDF only"]["oof"][0])
    display(evaluate_on_test(lambda: logistic_pipeline(["tfidf"]), thr).round(3).to_frame("test"))
else:
    print("Test evaluation is switched off.")